In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
import gc
import re

import matplotlib.pyplot as plt
import seaborn as sns
import numpy.random as random
from pandas import Series, DataFrame

import lightgbm as lgb
import sklearn
from sklearn.model_selection import cross_val_score
from sklearn.tree import  DecisionTreeRegressor

#????
!pip install ydata_profiling
import ydata_profiling as pdp

#???
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder,OneHotEncoder,OrdinalEncoder
#?????
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold,StratifiedGroupKFold
from sklearn.metrics import f1_score,recall_score, precision_score
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
dfff_train = pd.read_csv("/sumple/train.csv")
dfff_test = pd.read_csv("/sumple/test.csv")

dff_train = dfff_train.copy()
dff_test = dfff_test.copy()

In [ ]:
#?????????????????????
df_all = pd.concat([dff_train, dff_test], keys=["train","test"]).reset_index(level =0)

In [ ]:
#?????
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print("Memory usage of dataframe is {:.2f}MB".format(start_mem))

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
             pass

    end_mem = df.memory_usage().sum() / 1024**2
    print("Memory usage after optimization is: {:.2f} MB".format(end_mem))
    print("Decreased by {:.1f}%".format(100 * (start_mem - end_mem) / start_mem))

    return df

In [ ]:
#????????
df_all = reduce_mem_usage(df_all)
df_train = reduce_mem_usage(df_all)
df_test = reduce_mem_usage(df_all)

In [ ]:
#??????????????
df_all.head()

In [ ]:
#??????
df_all["ProdTaken"].value_counts()

In [ ]:
#?????
df_all.isna().sum()

In [ ]:
#????????
df_all["Age"] = df_all["Age"].astype(str)

In [ ]:
df_all["Age"].head()

In [ ]:
# ????????????????????????
table = str.maketrans({chr(0xFF10 + i): chr(0x30 + i) for i in range(10)})

# NaN?????????
for index, age in df_all["Age"].dropna().items():
    # age ?????????????????? translate ???
    if isinstance(age, (int, float)):
        normalized_age = str(age).translate(table)
    else:
        normalized_age = age.translate(table)
    df_all.loc[index, "Age"] = normalized_age
    df_all.loc[index, "Age_cleaned"] = normalized_age

In [ ]:
# ?????????????
for index, age in df_all["Age_cleaned"].items():
    cleaned_age = age.replace('?', '').replace('?', '').replace('?', '').replace('?', '')
    df_all.at[index, 'Age_cleaned'] = cleaned_age

df_all["Age_cleaned"].value_counts()

In [ ]:
#??????
def convert_kanji_to_number(text):
    kanji_to_number = {
        "?": 1, "?": 2, "?": 3, "?": 4, "?": 5, "?": 6, "?": 7, "?": 8, "?": 9,
        "?": 10
    }
    result = 0
    temp_value = 0
    for char in text:
        if char in kanji_to_number:
            value = kanji_to_number[char]
            if value == 10:
                result += temp_value * 10
                temp_value = 0
            else:
                temp_value += value
    result += temp_value
    return result if result > 0 else text

# ???? "?" ???
df_all['Age_numeric'] = df_all['Age_cleaned'].str.replace("?", "").apply(convert_kanji_to_number)

In [ ]:
df_all['Age_numeric'] = pd.to_numeric(df_all['Age_numeric'], errors='coerce')
df_all['Age_numeric'] = df_all['Age_numeric'].clip(lower=20, upper = 60)

In [ ]:
df_all["Age_numeric"].value_counts()

In [ ]:
# ????????:????????
df_all["TypeofContact"].value_counts()

In [ ]:
#???(1>2>3)
df_all["CityTier"].value_counts()

In [ ]:
#??????????????????
df_all["DurationOfPitch"].value_counts()

In [ ]:
df_all["DurationOfPitch"] = df_all["DurationOfPitch"].astype(str)
# ?????? "Pitch?" ???
df_all["Pitch?"] = df_all["DurationOfPitch"].copy().values

In [ ]:
# ????
df_all['Pitch?'] = df_all['DurationOfPitch'].str.replace('?', '')

# ????????????60???
df_all['Pitch_minutes'] = df_all['Pitch?'].apply(lambda x: int(x) / 60 if x.isdigit() else x)

In [ ]:
# ????
df_all['Pitch_minutes'] = df_all['Pitch_minutes'].astype(str).str.replace('?', '')

df_all["Pitch_minutes"] = df_all["Pitch_minutes"].astype(float)
df_all['Pitch_minutes'] = df_all['Pitch_minutes'].astype('Int64')
df_all["Pitch_minutes"] = df_all["Pitch_minutes"].replace('nan', np.nan)

In [ ]:
#?????
df_all["Pitch_minutes"].isnull().sum()

In [ ]:
#??????:??????????
df_all["Occupation"].value_counts()

In [ ]:
df_all["Gender"].value_counts()

In [ ]:
df_all["Gender"] = df_all["Gender"].astype(str)
def convert_gender(gender):
  gender = gender.lower().replace(" ", "")  # ?????????????
  return gender  # ??????????

df_all["Gender_enc"] = df_all["Gender"].apply(convert_gender)


In [ ]:
import unicodedata

df_all["Gender_enc"] = df_all["Gender_enc"].astype(str)

def convert_to_binary(gender):
    # ????????
    gender = unicodedata.normalize('NFKC', gender)
    # ??????????????
    gender = gender.translate(str.maketrans({chr(0xFF01 + i): chr(0x21 + i) for i in range(94)}))

    # ??????
    gender = gender.lower()

    # ????: "female" ??? "f" ??????????
    if "female" in gender or "f" in gender:
        return 0
    else:
        return 1

df_all["Gender_enc"] = df_all["Gender_enc"].apply(convert_to_binary)

In [ ]:
display(df_all[["Gender", "Gender_enc"]].sample(n=20))
# female?1

In [ ]:
#?????????????????
df_all["NumberOfPersonVisiting"].value_counts()

In [ ]:
df_all["NumberOfFollowups"].isnull().sum()

In [ ]:
#???????????????
df_all["PreferredPropertyStar"].value_counts()

In [ ]:
#???????????????
df_all["ProductPitched"].value_counts()

In [ ]:
df_all["ProductPitched"] = df_all["ProductPitched"].astype(str)

def convert_gender_to_binary(product):
  # ???????????????????????
  product = unicodedata.normalize('NFKC', product).lower().replace(" ", "").replace("|", "l")

  return product

df_all["ProductPitched_low"] = df_all["ProductPitched"].apply(convert_gender_to_binary)

In [ ]:
def normalize_text(text):
  # ?????????????????????
  normalized_text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8').lower()
  return normalized_text

df_all["ProductPitched_low"] = df_all["ProductPitched_low"].apply(normalize_text)

In [ ]:
def replace_multiple(text):
    # ???????basic??
    text = re.sub(r"^(ba?s?i?c?)+$", "basic", text)

    return text

df_all["ProductPitched_low"] = df_all["ProductPitched_low"].apply(replace_multiple)

In [ ]:
#basic??
def replace_multiple(text):
    if text in ["asc", "asic", "asi"]:
        text = "basic"
    return text

df_all["ProductPitched_low"] = df_all["ProductPitched_low"].apply(replace_multiple)

In [ ]:
#king
def replace_multiple(text):
    if text in ["kig", "kng", "kg"]:
        text = "king"
    return text

df_all["ProductPitched_low"] = df_all["ProductPitched_low"].apply(replace_multiple)

In [ ]:
#deluxe
def replace_multiple(text):
    if text in ["eluxe","delue", "delux","dlux"]:
        text = "deluxe"
    return text

df_all["ProductPitched_low"] = df_all["ProductPitched_low"].apply(replace_multiple)

In [ ]:
#superdeluxe
def replace_multiple(text):
    if text in ["supereluxe","uperdeluxe", "superdluxe", "superdelue","supreluxe"]:
        text = "superdeluxe"
    return text

df_all["ProductPitched_low"] = df_all["ProductPitched_low"].apply(replace_multiple)

In [ ]:
#standard
def replace_multiple(text):
    if text in ["tandard","stadard","stndard","standar","stanard"]:
        text = "standard"
    return text

df_all["ProductPitched_low"] = df_all["ProductPitched_low"].apply(replace_multiple)

In [ ]:
df_all["ProductPitched_low"].value_counts()

In [ ]:
#????????
df_all["NumberOfTrips"].value_counts()

In [ ]:
df_all["NumberOfTrips"] = df_all["NumberOfTrips"].astype(str)
def extract_trips(trips):
    if trips == "????1?" or trips == "???1?":
        return trips  # "????1?" ? "???1?" ???????

    trips = trips.strip()  # trips ????????????

    original_trips = trips  # ??????

    # "??" ? "?" ???
    trips = re.sub(r"???", "", trips)
    trips = re.sub(r"??", "", trips)

    if trips.isdigit():  # trips ????????????????
        return int(trips)  # ?????????????????

    return original_trips  # ??????????????

df_all["NumberOfTrips_year"] = df_all["NumberOfTrips"].apply(extract_trips)

In [ ]:
df_all["NumberOfTrips_year"].value_counts()

In [ ]:
#??Number???
df_all["NumberOfTrips_year"] = df_all["NumberOfTrips_year"].astype(str)
def replace_multiple(text):
    if text in ["???1?"]:
        text = 2
    elif text in ["????1?"]:
        text = 4
    elif text == "nan":
        text = 0
    return text

df_all["NumberOfTrips_year"] = df_all["NumberOfTrips_year"].apply(replace_multiple)
df_all["NumberOfTrips_year"] = df_all["NumberOfTrips_year"].astype(int)
df_all["NumberOfTrips_year"].value_counts()

In [ ]:
df_all["Passport"].value_counts()

In [ ]:
#???????????
df_all["PitchSatisfactionScore"].value_counts()

In [ ]:
# ??????????????
df_all["Designation"].value_counts()

In [ ]:
df_all["Designation"] = df_all["Designation"].astype(str)

def convert_gender_to_binary(product):
  # ???????????????????????
  product = unicodedata.normalize('NFKC', product).lower().replace(" ", "").replace("|", "l")

  return product

df_all["Designation_low"] = df_all["Designation"].apply(convert_gender_to_binary)

In [ ]:
def normalize_text(text):
  # ?????????????????????
  normalized_text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8').lower()
  return normalized_text

df_all["Designation_low"] = df_all["Designation_low"].apply(normalize_text)

In [ ]:
#executive
def replace_multiple(text):
    if text in ["excutive", "excutiv", "executiv","executie","xecutive",
                "eecutive", "excuti", "xecutiv", "executi", "cutive", "ecutive", "xcutiv","ecutie" ]:
        text = "executive"
    return text

df_all["Designation_low"] = df_all["Designation_low"].apply(replace_multiple)

In [ ]:
#manager
def replace_multiple(text):
    if text in ["anager", "managr", "manger", "mnager", "mnger", "nager",
                "mnagr", "mangr", "anagr", "nagr", "mngr" ]:
        text = "manager"
    return text

df_all["Designation_low"] = df_all["Designation_low"].apply(replace_multiple)

In [ ]:
#seniormanager
def replace_multiple(text):
    if text in ["seniormanagr", "sniormanager", "eniormanager", "seniormanger", "seniormnger", "enioranager",
                "sniormanagr", "seniormnager", "senioranager", "senioranger", "eniormanger" ]:
        text = "seniormanager"
    return text

df_all["Designation_low"] = df_all["Designation_low"].apply(replace_multiple)

In [ ]:
#vip
def replace_multiple(text):
    if text in ["avp", "vp" ]:
        text = "vip"
    return text

df_all["Designation_low"] = df_all["Designation_low"].apply(replace_multiple)

In [ ]:
df_all["Designation_low"].value_counts()

In [ ]:
#?????
df_all["MonthlyIncome"].head()

In [ ]:
# df_all["MonthlyIncome"] ?str????
df_all["MonthlyIncome"] = df_all["MonthlyIncome"].astype(str)

# ?????????????
df_all["MonthlyIncome_ext"] = df_all["MonthlyIncome"].str.extract(r"??(\d+\.?\d*)??")

# ?????????????10000????
df_all["MonthlyIncome_ext"] = np.where(df_all["MonthlyIncome_ext"].notnull(), df_all["MonthlyIncome_ext"].astype(float) * 10000, df_all["MonthlyIncome"])

In [ ]:
df_all["MonthlyIncome"].head()

In [ ]:
# ?????? ("nan" ? NaN ???)
df_all["MonthlyIncome_ext"] = pd.to_numeric(df_all["MonthlyIncome_ext"], errors='coerce')

In [ ]:
#????????(?????????????????????????????????)
df_all["customer_info"].value_counts()

In [ ]:
df_all["child_info"] = df_all["customer_info"].str.replace(r"[\s()]", "", regex=True)
child_info_temp1 = df_all["child_info"].str.extract(r"(??.+|???.+)")

child_info_temp3 = child_info_temp1.copy()
child_info_temp2 = df_all["customer_info"].str.extract(r"(1?|2?|3?|??|??|?????|???????|?????|?????)\s?")
df_all["child_info"] = child_info_temp3[0].fillna(child_info_temp2[0])

In [ ]:
# "customer_info" ??  "customer_info_removed" ???
df_all["customer_info_removed"] = df_all["customer_info"].str.replace(r"(??|???|1?|2?|3?|?????|??|??|?????|???????|?????).*\s?", "", regex=True)

In [ ]:
df_all["child_info"].value_counts()
df_all["child_info"].isna().sum()

In [ ]:
display(df_all[["customer_info", "customer_info_removed","child_info"]].sample(n=10))

In [ ]:
df_all["customer_info_removed"].value_counts()

In [ ]:
df_all["material_status"] = df_all["customer_info_removed"].str.extract(r"(????|??|????|??|)")

In [ ]:
df_all["customer_info_rem2"] = df_all["customer_info_removed"].str.replace(r"(????|??|????|??)", "", regex=True)

In [ ]:
df_all["material_status"].value_counts()

In [ ]:
df_all["customer_info_rem2"].value_counts()

In [ ]:
df_all["car_status"] = df_all["customer_info_removed"].str.extract(r"(??|???|???|??|??|??|????|??)")

In [ ]:
df_all["car_status"].isna().sum()

In [ ]:
display(df_all[["customer_info", "child_info","material_status", "car_status"]].sample(n=10))

In [ ]:
df_all["child_info"].value_counts()

In [ ]:
# child_info_int ?????
df_all['child_info_int'] = df_all['child_info'].copy()

# ???????????????
df_all['child_info_int'] = pd.to_numeric(df_all['child_info_int'].str.extract(r'(\d+)')[0], errors='coerce')

# ????????????????????
df_all['child_info_int'] = df_all['child_info_int'].fillna(df_all['child_info'])

In [ ]:
df_all["child_info_cat"] =df_all["child_info_int"].copy()

In [ ]:
df_all["child_info_int"].value_counts()

In [ ]:
def convert_child_info(value):
  if value in ["????", "????","????", "??", "?????","??????", "??","???????", "?????", "?????"]:
    return 0
  else:
    return value

df_all["child_info_int"] = df_all["child_info_int"].apply(convert_child_info)

In [ ]:
df_all["child_info_int"].value_counts()

In [ ]:
def convert_child_info1(value):
  if value in ["????", "????","????", "??", "?????"]:
    return 0
  elif value in ["??????", "??","???????", "?????", "?????"]:
    return -1
  else:
    return value

df_all["child_info_cat"] = df_all["child_info_cat"].apply(convert_child_info1)

In [ ]:
df_all["child_info_cat"].value_counts()

In [ ]:
# #????????????????????????
df_all['material_count'] = df_all['material_status'].copy()

def convert_material_info(value):
  if value in ["????"]:
    return 2
  elif value in ["????"]:
    return 1
  elif value in ["??", "??"]:
    return 0
  else:
    return value

df_all["material_count"] = df_all["material_count"].apply(convert_material_info)

In [ ]:
df_all["material_status"].value_counts()

In [ ]:
df_all["material_count"].value_counts()

In [ ]:
df_all["car_status"].value_counts()

In [ ]:
# ???????????
df_all['car_status_count'] = df_all['car_status'].copy()

def convert_car_info(value):
  if value in ["??","??", "??", "??"]:
    return 1
  elif value in ["???", "???", "??","????", ""]:
    return 0
  else:
    return value

df_all["car_status_count"] = df_all["car_status_count"].apply(convert_car_info)

In [ ]:
df_all["car_status_count"].value_counts()

In [ ]:
df_all.isnull().sum()

In [ ]:
#????
df_all.info()

In [ ]:
#???????
df_all["Passport"] = df_all["Passport"].astype(int)
df_all["PitchSatisfactionScore"] = df_all["PitchSatisfactionScore"].astype(int)

In [ ]:
for col in df_all.columns:
  if df_all[col].dtype == 'float16':
    df_all[col] = df_all[col].astype('float64')

In [ ]:
for col in df_all.columns:
  if df_all[col].dtype == "int8":
    df_all[col] = df_all[col].astype("int64")

In [ ]:
for col in df_all.columns:
  if df_all[col].dtype == "object":
    df_all[col] = df_all[col].astype("category")

In [ ]:
for col in df_all.columns:
  if df_all[col].dtype == "int16":
    df_all[col] = df_all[col].astype("int64")

In [ ]:
#??????????
df_all0 = df_all.copy()